# Conditional 3D prediction

Generate one categorical volume from the trained L-MPDD model, then improve the same latent with online critic guidance, exact decoded anchor losses, phase-fraction guidance, and final quality gates.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
import time

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app.api import AnchorSlice, PredictOptions
from src.app.runtime import (
    build_dataset,
    build_loader,
    load_defaults,
    load_predict_config,
    load_predictor,
)
from src.modeling.phases import quantize_phase


def take_slice(volume, axis, index):
    return np.take(volume, index, axis=axis)


## Parameters

The two center anchors are compatible at their intersection. Replace either image independently when using real multi-axis conditions.

In [ ]:
RUN_DIR = "run/20260712-163751-714469"
PHASE_FRACTIONS = (0.28, 0.12, 0.60)
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"

## Predict

This is the only base-size generation path: L-MPDD latent sampling, optional critic warm-up, joint latent residual refinement, categorical Refine candidates, calibration, and selection.

In [ ]:
run_dir = ROOT / RUN_DIR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = Namespace(**load_defaults(run_dir / "vae.yaml"))
args.data_dir = ROOT / args.data_dir
args.batch_size = 1

batch = next(build_loader(build_dataset(args), args, device=torch.device("cpu")))
anchor_image = quantize_phase(batch[0, 0], args.num_phases).numpy()
center_index = anchor_image.shape[0] // 2
anchors = [
    AnchorSlice(image=anchor_image, axis=0, index=center_index),
    AnchorSlice(image=anchor_image.copy(), axis=1, index=center_index),
]
options = PredictOptions(
    num_phases=args.num_phases,
    phase_fractions=PHASE_FRACTIONS,
    **load_predict_config(PREDICT_CONFIG),
)

predictor = load_predictor(run_dir, device=device)
start_time = time.perf_counter()
volume, stats = predictor.predict(options, anchors=anchors)
elapsed_seconds = time.perf_counter() - start_time
volume_np = volume.cpu().numpy()

print("device:", device)
print(f"elapsed: {elapsed_seconds:.1f} seconds")
print("volume:", volume_np.shape, volume.dtype)
print("anchors:", [(anchor.axis, anchor.index) for anchor in anchors])
print("candidates:", int(stats["candidate_count"]))
print("selected latent step:", int(stats["selected_latent_step"]))
print("selected refine steps:", int(stats["selected_refine_steps"]))
print("anchor mismatches:", np.round(stats["final_anchor_mismatches"].cpu().numpy(), 4).tolist())
print("target phase fraction:", np.round(stats["final_target_phase_fraction"].cpu().numpy(), 4).tolist())
print("volume phase fraction:", np.round(stats["final_phase_fraction"].cpu().numpy(), 4).tolist())

## Quality check

Inspect anchor similarity, requested phase fractions, axis balance, repeated slices, and continuity. Anchors are optimized softly and are never copied into the output.

In [ ]:
anchor_mismatches = stats["final_anchor_mismatches"].cpu().numpy()
volume_phase_fraction = stats["final_phase_fraction"].cpu().numpy()
axis_transition_rate = stats["final_axis_transition_rate"].cpu().numpy()
axis_repeat_rate = stats["final_axis_exact_repeat_rate"].cpu().numpy()
axis_boundary_jump = stats["final_axis_global_boundary_jump"].cpu().numpy()
axis_run_mae = stats["final_axis_run_profile_mae"].cpu().numpy()

quality_gates = {
    "anchor mismatch within tolerance": bool(np.max(anchor_mismatches) <= options.quality.anchor_tolerance),
    "phase fraction within tolerance": bool(np.max(np.abs(volume_phase_fraction - PHASE_FRACTIONS)) <= options.phase_fraction_tolerance),
    "axis transition spread within tolerance": bool(np.ptp(axis_transition_rate) <= options.quality.morphology_tolerance),
    "no repeated adjacent planes": bool(np.max(axis_repeat_rate) <= options.quality.repeat_tolerance),
    "boundary jump within tolerance": bool(np.max(axis_boundary_jump) <= options.quality.continuity_tolerance),
    "run-profile error within tolerance": bool(np.max(axis_run_mae) <= options.quality.morphology_tolerance),
}

print("axis transition rate:", np.round(axis_transition_rate, 4).tolist())
print("axis run-profile MAE:", np.round(axis_run_mae, 4).tolist())
print("exact repeated-slice rate:", np.round(axis_repeat_rate, 4).tolist())
print("global boundary jump:", np.round(axis_boundary_jump, 4).tolist())
print("calibration changed fraction:", round(float(stats["calibration_changed_fraction"]), 4))
print("quality gates:", quality_gates)

fig, axes = plt.subplots(len(anchors), 3, figsize=(11, 3.6 * len(anchors)), squeeze=False)
for row, anchor in enumerate(anchors):
    generated = take_slice(volume_np, anchor.axis, anchor.index)
    difference = generated != anchor.image
    panels = [
        (anchor.image, f"condition axis={anchor.axis}", "gray", 0, args.num_phases - 1),
        (generated, f"generated index={anchor.index}", "gray", 0, args.num_phases - 1),
        (difference, f"different {anchor_mismatches[row]:.1%}", "magma", 0, 1),
    ]
    for axis, (image, title, cmap, vmin, vmax) in zip(axes[row], panels):
        axis.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        axis.set_title(title)
        axis.axis("off")
plt.tight_layout()

indices = np.linspace(0, volume_np.shape[0] - 1, 9).round().astype(int)
fig, axes = plt.subplots(3, len(indices), figsize=(18, 6), squeeze=False)
for row, axis_id in enumerate(range(3)):
    for column, index in enumerate(indices):
        axes[row, column].imshow(
            take_slice(volume_np, axis_id, index),
            cmap="gray",
            vmin=0,
            vmax=args.num_phases - 1,
            interpolation="nearest",
        )
        axes[row, column].set_title(f"axis {axis_id} / {index}")
        axes[row, column].axis("off")
plt.tight_layout()

## 3D slice browser

Choose an axis and move the slider to inspect every generated slice.

In [ ]:
axis_selector = widgets.ToggleButtons(
    options=[("XY", 0), ("XZ", 1), ("YZ", 2)],
    value=0,
    description="axis",
)
index_slider = widgets.IntSlider(
    value=volume_np.shape[0] // 2,
    min=0,
    max=volume_np.shape[0] - 1,
    step=1,
    description="index",
    continuous_update=False,
)


def show_slice(axis, index):
    plt.figure(figsize=(5, 5))
    plt.imshow(
        take_slice(volume_np, axis, index),
        cmap="gray",
        vmin=0,
        vmax=args.num_phases - 1,
        interpolation="nearest",
    )
    plt.title(f"axis {axis} / index {index}")
    plt.axis("off")
    plt.show()


slice_output = widgets.interactive_output(
    show_slice,
    {"axis": axis_selector, "index": index_slider},
)
display(widgets.HBox([axis_selector, index_slider]), slice_output)
